In [1]:
"""
Bai06 - Lab thực hành Chương 13: Viết và Bảo trì Kiểm thử Đơn vị với GenAI
Sách: Supercharged Coding with Gen AI

Mục tiêu:
  - Viết unit test với pytest / unittest cho hàm xử lý n-gram
  - Minh họa data-driven tests (parametrize)
  - Minh họa TDD: viết test trước, code sau

Prompt mẫu dùng với Copilot/ChatGPT:
  "@workspace /tests generate unit tests for create_ngrams function
   including edge cases: empty string, single char, n > len(text)"
"""

import re
import unittest
import pytest


# ─────────────────────────────────────────────
# Implementation code (code cần test)
# ─────────────────────────────────────────────
def lowercase_remove_punct_numbers(text: str) -> str:
    """Convert text to lowercase and remove punctuation/numbers.

    Args:
        text (str): Input text string.

    Returns:
        str: Cleaned text with only lowercase letters and whitespace.
    """
    return re.sub(r'[^a-z\s]', '', text.lower())


def multiple_to_single_spaces(text: str) -> str:
    """Replace multiple consecutive spaces with a single space.

    Args:
        text (str): Input text string.

    Returns:
        str: Text with normalized spacing.
    """
    return re.sub(r'\s+', ' ', text).strip()


def create_ngrams(text: str, n: int) -> list:
    """Create a list of n-gram strings from input text.

    Processes the text by converting to lowercase, removing punctuation
    and numbers, then generating all n-grams.

    Args:
        text (str): Input text to generate n-grams from.
        n (int): Length of each n-gram.

    Returns:
        list: List of n-gram strings.

    Raises:
        ValueError: If n <= 0.
    """
    if n <= 0:
        raise ValueError("n must be a positive integer.")

    processed = lowercase_remove_punct_numbers(text)
    processed = multiple_to_single_spaces(processed)

    if n > len(processed):
        return []

    return [processed[i:i + n] for i in range(len(processed) - n + 1)]


# ─────────────────────────────────────────────
# Unit Tests – unittest (GenAI Generated)
# ─────────────────────────────────────────────
class TestLowercaseRemovePunct(unittest.TestCase):
    """Tests for lowercase_remove_punct_numbers function."""

    def test_removes_uppercase(self):
        self.assertEqual(lowercase_remove_punct_numbers("Hello"), "hello")

    def test_removes_punctuation(self):
        self.assertEqual(lowercase_remove_punct_numbers("hello!"), "hello")

    def test_removes_numbers(self):
        self.assertEqual(lowercase_remove_punct_numbers("abc123"), "abc")

    def test_preserves_spaces(self):
        self.assertEqual(lowercase_remove_punct_numbers("hello world"), "hello world")

    def test_empty_string(self):
        self.assertEqual(lowercase_remove_punct_numbers(""), "")

    def test_only_special_chars(self):
        self.assertEqual(lowercase_remove_punct_numbers("$%^&*"), "")


class TestMultipleToSingleSpaces(unittest.TestCase):
    """Tests for multiple_to_single_spaces function."""

    def test_double_space(self):
        self.assertEqual(multiple_to_single_spaces("a  b"), "a b")

    def test_tab_to_space(self):
        result = multiple_to_single_spaces("a\tb")
        self.assertEqual(result, "a b")

    def test_already_single_space(self):
        self.assertEqual(multiple_to_single_spaces("a b"), "a b")

    def test_strips_leading_trailing(self):
        self.assertEqual(multiple_to_single_spaces("  hello  "), "hello")


class TestCreateNgrams(unittest.TestCase):
    """Tests for create_ngrams function."""

    def test_basic_3gram(self):
        result = create_ngrams("abcde", 3)
        self.assertEqual(result, ["abc", "bcd", "cde"])

    def test_n_equals_length(self):
        result = create_ngrams("abc", 3)
        self.assertEqual(result, ["abc"])

    def test_n_greater_than_length(self):
        result = create_ngrams("ab", 5)
        self.assertEqual(result, [])

    def test_empty_string(self):
        result = create_ngrams("", 3)
        self.assertEqual(result, [])

    def test_invalid_n(self):
        with self.assertRaises(ValueError):
            create_ngrams("hello", 0)

    def test_removes_special_chars(self):
        result = create_ngrams("$ABC%", 3)
        self.assertEqual(result, ["abc"])

    def test_real_sentence(self):
        result = create_ngrams("Hi!", 2)
        self.assertEqual(result, ["hi"])


# ─────────────────────────────────────────────
# Data-Driven Tests – pytest parametrize
# ─────────────────────────────────────────────
@pytest.mark.parametrize("text,n,expected", [
    ("abcde", 2, ["ab", "bc", "cd", "de"]),
    ("hello", 5, ["hello"]),
    ("hi", 3, []),
    ("", 2, []),
    ("ABC", 2, ["ab", "bc"]),
    ("a b", 2, ["a ", " b"]),
])
def test_create_ngrams_parametrized(text, n, expected):
    """Data-driven test for create_ngrams with multiple cases."""
    assert create_ngrams(text, n) == expected


@pytest.mark.parametrize("text,expected", [
    ("Hello World!", "hello world"),
    ("Python3.11", "python"),
    ("  spaces  ", "spaces"),
    ("", ""),
])
def test_cleanup_pipeline(text, expected):
    """Test full text cleanup pipeline."""
    result = multiple_to_single_spaces(lowercase_remove_punct_numbers(text))
    assert result == expected


# ─────────────────────────────────────────────
# TDD Demo: test trước, sau đó viết code
# ─────────────────────────────────────────────
def count_unique_ngrams(text: str, n: int) -> int:
    """Count the number of unique n-grams in text.

    (Hàm này được viết SAU khi có test TDD bên dưới)

    Args:
        text (str): Input text.
        n (int): N-gram length.

    Returns:
        int: Number of unique n-grams.
    """
    return len(set(create_ngrams(text, n)))


class TestCountUniqueNgrams(unittest.TestCase):
    """TDD: Tests written BEFORE implementation."""

    def test_all_unique(self):
        # "abcde" -> 3-grams: abc, bcd, cde (all unique)
        self.assertEqual(count_unique_ngrams("abcde", 3), 3)

    def test_with_duplicates(self):
        # "abab" -> 3-grams: aba, bab, aba → 2 unique
        self.assertEqual(count_unique_ngrams("abab", 3), 2)

    def test_empty(self):
        self.assertEqual(count_unique_ngrams("", 3), 0)


# ─────────────────────────────────────────────
# Demo chạy
# ─────────────────────────────────────────────
if __name__ == "__main__":
    print("=" * 50)
    print("Ch13 Lab: Unit Tests with GenAI")
    print("=" * 50)

    text = "This is a sample text $ABC% for creating n-grams."
    n = 3
    result = create_ngrams(text, n)
    print(f"\nInput: '{text}'")
    print(f"3-grams ({len(result)} total): {result[:10]}...")
    print(f"Unique 3-grams: {count_unique_ngrams(text, n)}")

    print("\n[Running unittest]")
    loader = unittest.TestLoader()
    suite = unittest.TestSuite()
    suite.addTests(loader.loadTestsFromTestCase(TestCreateNgrams))
    runner = unittest.TextTestRunner(verbosity=2)
    runner.run(suite)

    print("\n[Hint] Run pytest for parametrized tests:")
    print("  pytest ch13_unit_test_demo.py -v")


test_basic_3gram (__main__.TestCreateNgrams) ... ok
test_empty_string (__main__.TestCreateNgrams) ... ok
test_invalid_n (__main__.TestCreateNgrams) ... ok
test_n_equals_length (__main__.TestCreateNgrams) ... ok
test_n_greater_than_length (__main__.TestCreateNgrams) ... ok
test_real_sentence (__main__.TestCreateNgrams) ... ok
test_removes_special_chars (__main__.TestCreateNgrams) ... ok

----------------------------------------------------------------------
Ran 7 tests in 0.009s

OK


Ch13 Lab: Unit Tests with GenAI

Input: 'This is a sample text $ABC% for creating n-grams.'
3-grams (43 total): ['thi', 'his', 'is ', 's i', ' is', 'is ', 's a', ' a ', 'a s', ' sa']...
Unique 3-grams: 42

[Running unittest]

[Hint] Run pytest for parametrized tests:
  pytest ch13_unit_test_demo.py -v


In [2]:
# ─────────────────────────────────────────────
# Additional Unit Tests – lowercase_remove_punct_numbers
# ─────────────────────────────────────────────

class TestLowercaseRemovePunctMore(unittest.TestCase):
    """More edge-case tests for lowercase_remove_punct_numbers."""

    def test_preserves_newlines_and_tabs(self):
        self.assertEqual(
            lowercase_remove_punct_numbers("Hello\nWorld\tOK"),
            "hello\nworld\tok",
        )

    def test_removes_common_separators(self):
        self.assertEqual(
            lowercase_remove_punct_numbers("snake_case-kebab-case's"),
            "snakecasekebabcases",
        )

    def test_keeps_only_whitespace_when_no_letters(self):
        # Digits/punct are removed but whitespace remains
        self.assertEqual(lowercase_remove_punct_numbers("123 456"), " ")

    def test_removes_non_ascii_letters(self):
        # Function keeps only a-z; accented letters are removed
        self.assertEqual(lowercase_remove_punct_numbers("Café"), "caf")


@pytest.mark.parametrize(
    "text,expected",
    [
        ("ABC", "abc"),
        ("Hello, WORLD!!!", "hello world"),
        ("mix3d numb3rs", "mixd numbrs"),
        ("  Leading and  trailing  ", "  leading and  trailing  "),
        ("$%^&*", ""),
        ("line1\nline2", "line\nline"),
    ],
)
def test_lowercase_remove_punct_numbers_parametrized(text, expected):
    assert lowercase_remove_punct_numbers(text) == expected


In [ ]:

def test_phone_number_10_digits(self):
    self.assertEqual(
        create_ngrams("(123) 456-7890", 2), ["12", "23", "34", "45", "56", "67", "78", "89", "90"]
    )